In [3]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
import math as m
from math import pi, degrees, radians
import os

Velocidad_Vuelo = int(input('Ingrese la velocidad de vuelo en [m/s]. En caso de ser descenso vertical, ingrese 0:'))

Ingrese la velocidad de vuelo en [m/s]. En caso de ser descenso vertical, ingrese 0: 35


In [5]:
#.Coordenadas
rR   = 4.2
Nseg = 10
Nnod = Nseg + 1
Inc  = 0.2
strt = Inc
stop = rR
Rnod = np.linspace(strt, stop, Nnod)
rBl  = 0.5 * (Rnod[0:Nseg] + Rnod[1:Nnod])

convGrad = 180/pi
convRad = pi/180

In [7]:
#if (Velocidad_Vuelo == 0):
#.Datos del vuelo en descenso del autogiro
g    = 9.81                     #[m/s^2]. Gravedad
ma   = 392                      #[kg].    Masa del autogiro
mp   = 12.8                     #[kg].    Masa de una pala
P    = ma * g                   #[N].     Peso del rotor
nR   = 311                      #[rpm].   Velocidad rotor
Vsink= 7.7                      #[m/s].   Velocidad descenso
rR   = 4.2                      #[m]      Radio del Rotor.
SR   = pi * rR**2               #[m^2]    Area del Rotor.
ΩR   = nR * (pi / 30)           #[rads/s] Velocidad Angular
rho  = 1.225                    #.        Densidad del aire
VS   = nR * rR                  #[m/s]    Velocidad en punta de pala
C    = 0.2                      #[m].     Cuerda de la pala

In [9]:
VBl = ΩR * rBl

#.Angulo de Flujo
wRi = 6.5                     #.Velocidad flujo inducido
wBl = Vsink - wRi             #.Velocidad del flujo
ϕBl = (wBl / VBl) * 60        #.Angulo de Flujo
ϵBl = 2.5                     #.Ángulo de pala
αBl = ϕBl + ϵBl               #.Ángulo de ataque de la pala

In [11]:
    #.Fuerza del aire sobre las palas del rotor
    Cd = 0.007
    
    def Clperfil(Alpha):
        Cl = 0.24 + 2*pi * Alpha * convRad
        return Cl
        
    def Sustentacion(rho, V, Cl, A):
        L = 0.5 * rho * V**2 * Cl * A
        return L
        
    def Resistencia(rho, V, Cd, A):
        R = 0.5 * rho * V**2 * Cd * A
        return R
        
    def FuerzaParalela(ABl, ϕBl, WBl):
        FP = ABl * (ϕBl/60) - WBl
        return FP

    def FuerzaTotal(FP, FT):
        FT = np.sqrt(sum(FXBl)**2 + sum(FZBl)**2)
        return FT

    def Par(FXBl, rBl):
        M = FXBl * rBl  
        return M
        
    SBl  = (rR/Nseg) * C
    CABl = Clperfil(αBl)
    ABl  = Sustentacion(rho, VBl, CABl, SBl)     #.Sustentación
    WBl  = Resistencia(rho, VBl, Cd, SBl)        #.Arrastre
    
    FXBl = FuerzaParalela(ABl,ϕBl, WBl)          #.Fuerza Paralela
    FZBl = ABl                                   #.Fuerza tangencial
    FR   = FuerzaTotal(FXBl, FZBl)
    FRT  = 2 * np.sum(FR)
    MDBl = Par(FXBl, rBl)                        #.Par

In [13]:
    Datos_Descenso = pd.DataFrame({
        'rBl[m]'  : rBl,
        'VBl[m/s]': VBl,
        'ϕBl[º]'  : ϕBl,
        'αBl[º]'  : αBl,
        'CABl[]'  : CABl,
        'WBl[]'   : WBl,
        'FXBl[N]' : FXBl,
        'FZBl[N]' : FZBl,
        'MDBl[Nm]': MDBl
    })

    print('Datos del autogiro en Descenso Vertical.')
    display(Datos_Descenso)

Datos del autogiro en Descenso Vertical.


,rBl[m],VBl[m/s],ϕBl[º],αBl[º],CABl[],WBl[],FXBl[N],FZBl[N],MDBl[Nm]
0,0.4,13.027138,5.526924,8.026924,1.120251,0.061120,0.839893,9.781346,0.335957
1,0.8,26.054275,2.763462,5.263462,0.817203,0.244479,1.070067,28.541279,0.856053
2,1.2,39.081413,1.842308,4.342308,0.716187,0.550078,1.178001,56.279799,1.413601
3,1.6,52.108550,1.381731,3.881731,0.665679,0.977916,1.163696,92.996906,1.861914
4,2.0,65.135688,1.105385,3.605385,0.635375,1.527993,1.027152,138.692601,2.054303
5,2.4,78.162825,0.921154,3.421154,0.615172,2.200310,0.768368,193.366883,1.844083
6,2.8,91.189963,0.789561,3.289561,0.600741,2.994867,0.387344,257.019752,1.084564
7,3.2,104.217100,0.690866,3.190866,0.589918,3.911663,-0.115918,329.651209,-0.370939
8,3.6,117.244238,0.614103,3.114103,0.581500,4.950698,-0.741421,411.261253,-2.669115
9,4.0,130.271375,0.552692,3.052692,0.574765,6.111973,-1.489163,501.849884,-5.956650


In [15]:
#if (Velocidad_Vuelo != 0):
    # Datos de vuelo hacia adelante
ΨBl = np.array(np.arange(0, 360, 36))
SBl = 0.08
ΩR_Avance = 34                    #rad/seg
βBlc = 2.4
V = Velocidad_Vuelo               # m/s
v = 1.46E-5                    #m^2/s .viscosidad cinematica del aire a -15 ºC
a = 340                           #m/s velocidad del sonido a al nivel del mar y a 15➦C
rho = 1.225
ALPHA_OBJETIVO = 6.5

   #velocidad del flujo perpendicular
def wRi():                        #Velocidad de flujo inducido
    return 1.8
    
def αR():                         #Ángulo de ataque
    return 6.5
        
def wR(V):                        #Velocidad del flujo perpendicular
    return αR() * V * convRad

def Reynolds(V, C, v):
    return (abs(V) * C) / v / 1e6

In [17]:
ALPHA_OBJETIVO = 6.5

archivos_reynolds = [
    (0.030, "NACA 8-H-12 AIRFOIL_T1_Re0.030_M0.00_N9.0.txt"),
    (0.040, "NACA 8-H-12 AIRFOIL_T1_Re0.040_M0.00_N9.0.txt"),
    (0.060, "NACA 8-H-12 AIRFOIL_T1_Re0.060_M0.00_N9.0.txt"),
    (0.080, "NACA 8-H-12 AIRFOIL_T1_Re0.080_M0.00_N9.0.txt"),
    (0.100, "NACA 8-H-12 AIRFOIL_T1_Re0.100_M0.00_N9.0.txt"),
    (0.130, "NACA 8-H-12 AIRFOIL_T1_Re0.130_M0.00_N9.0.txt"),
    (0.160, "NACA 8-H-12 AIRFOIL_T1_Re0.160_M0.00_N9.0.txt"),
    (0.200, "NACA 8-H-12 AIRFOIL_T1_Re0.200_M0.00_N9.0.txt"),
    (0.300, "NACA 8-H-12 AIRFOIL_T1_Re0.300_M0.00_N9.0.txt"),
    (0.500, "NACA 8-H-12 AIRFOIL_T1_Re0.500_M0.00_N9.0.txt"),
    (1.000, "NACA 8-H-12 AIRFOIL_T1_Re1.000_M0.00_N9.0.txt"),
    (3.000, "NACA 8-H-12 AIRFOIL_T1_Re3.000_M0.00_N9.0.txt")
]

def leer_archivo_txt(nombre_archivo):
    datos = {
        'alpha': [], 'CL': [], 'CD': [], 'CDp': [], 'Cm': [],
        'Top_Xtr': [], 'Bot_Xtr': [], 'Cpmin': [], 'Chinge': [], 'XCp': []
    }
    with open(nombre_archivo, 'r') as archivo:
        leer_datos = False
        for linea in archivo:
            if "alpha" in linea:
                leer_datos = True
                continue
            if leer_datos:
                if '-' * 7 in linea:
                    continue
                valores = linea.strip().split()
                if len(valores) >= len(datos):
                    try:
                        for i, clave in enumerate(datos.keys()):
                            datos[clave].append(float(valores[i]))
                    except ValueError:
                        continue
    for clave in datos:
        datos[clave] = np.array(datos[clave])
    return datos

def interpolar_por_reynolds(Re, alpha_objetivo=ALPHA_OBJETIVO):

    # Buscar los Reynolds más cercanos
    menores = [par for par in archivos_reynolds if par[0] <= Re]
    mayores = [par for par in archivos_reynolds if par[0] >= Re]

    if not menores or not mayores:
        raise ValueError("Reynolds fuera del rango disponible")
        
    Re_inf, arch_inf = menores[-1]
    Re_sup, arch_sup = mayores[0]

    datos_inf = leer_archivo_txt(arch_inf)
    datos_sup = leer_archivo_txt(arch_sup)

    # Buscar índices más cercanos al alpha deseado
    def valor_en_alpha(datos, variable):
        alpha = datos['alpha']
        valores = datos[variable]
        return np.interp(ALPHA_OBJETIVO, alpha, valores)

    CL_inf = valor_en_alpha(datos_inf, 'CL')
    CL_sup = valor_en_alpha(datos_sup, 'CL')
    CD_inf = valor_en_alpha(datos_inf, 'CD')
    CD_sup = valor_en_alpha(datos_sup, 'CD')
    Cm_inf = valor_en_alpha(datos_inf, 'Cm')
    Cm_sup = valor_en_alpha(datos_sup, 'Cm')

    # Interpolación en Reynolds
    def interp(Re):
        return lambda a, b: a + (Re - Re_inf) * (b - a) / (Re_sup - Re_inf)

    interp_func = interp(Re)

    CL = interp_func(CL_inf, CL_sup)
    CD = interp_func(CD_inf, CD_sup)
    Cm = interp_func(Cm_inf, Cm_sup)

    return {
        'Re': Re,
        'CL': CL,
        'CD': CD,
        'Cm': Cm
    }

In [19]:
resultados_avanza = []
resultados_retrocede = []

def Pala_Avanza(ΩR_Avance, r, V, αR, βBlc, SBl):
    VBl_A     = ΩR_Avance * r + V                  #Velocidad de Flujo
    Re_A      = Reynolds(VBl_A, C, v)
    resultado = interpolar_por_reynolds(Re_A)
    
    CL_A      = resultado['CL']
    CD_A      = resultado['CD']

    wBl_A  = V * (αR * convRad - βBlc * convRad) - (βBlc * convRad * ΩR_Avance * r) - wRi()  #Componente perpendicular al plano del rotor de la VBl
    ϕBl_A  = (wBl_A / VBl_A) * convGrad
    αBl_A  = ϕBl_A + 2.5
    ABl_A  = (rho / 2) * VBl_A ** 2 * SBl * CL_A
    WBl_A  = (rho / 2) * VBl_A ** 2 * SBl * CD_A
    Ms_A   = ABl_A * r                          
    FxBl_A = ABl_A * ( ϕBl_A * convRad - (1/80) )
    
    Ma_A   = VBl_A / a
    
    rm = (r - 0.4) + 0.4/2
    
    FF_A   = (mp / Nseg) * rm * (ΩR_Avance**2)
    MDBl_A = FxBl_A * r + FF_A*rm
    
    ΓR_A   = ABl_A / (pi *  rho * r * VBl_A)
    wΓ_A   = ΓR_A / (2 * pi * r)
    wri_A  = ( 1 / ((r / rR) * pi) ) * ( (FR/2) / (rho * pi * r ** 2) ) * (1 / VBl_A)
    
    return VBl_A, wBl_A, ϕBl_A, αBl_A, CL_A, CD_A, ABl_A, WBl_A, Ms_A, FxBl_A, MDBl_A, Re_A, Ma_A, FF_A, wri_A

def Pala_Retrocede(ΩR_Avance, r, V, αR, βBlc, SBl):
    VBl_R  = ΩR_Avance * r - V
    Re_R   = Reynolds(VBl_R, C, v)
    resultado = interpolar_por_reynolds(Re_R)
    
    CL_R = resultado['CL']
    CD_R = resultado['CD']
    
    wBl_R  = V * (αR * convRad - βBlc * convRad) + (βBlc * convRad * ΩR_Avance * r) - wRi()
    ϕBl_R  = (wBl_R / VBl_R) * convGrad
    αBl_R  = ϕBl_R + 2.5
    ABl_R  = (rho / 2) * VBl_R ** 2 * SBl * CL_R
    WBl_R  = (rho / 2) * VBl_R ** 2 * SBl * CD_R
    Ms_R   = ABl_R * r                                  #Momento de Impacto
    FxBl_R = ABl_R * ( ϕBl_R * convRad - (1/80) )       #Fuerza tangencial

    rm = (r - 0.4) + 0.4/2    
    FF_R   = (mp/Nseg) * rm * (ΩR_Avance**2)             #Fuerza Centrifuga
    MDBl_R = FxBl_R * r + FF_R * rm                         #Momento
    
    Ma_R   = VBl_R/a                                    #Numero de Mach
    
    
    ΓR_R   = ABl_R / (pi *  rho * r * VBl_R)                                                    #Circulación                    
    wΓ_R   = ΓR_R / (2 * pi * r)                                                                #Velocidad inducida
    wri_R  = ( 1 / ((r / rR) * pi) ) * ( (FR/2) / (rho * pi * r**2) ) * (1 / VBl_R)          #Velocidad descendente inducida
    
    return VBl_R, wBl_R, ϕBl_R, αBl_R, CL_R, CD_R, ABl_R, WBl_R, Ms_R, FxBl_R, MDBl_R, Re_R, Ma_R, FF_R, wri_R

ri = np.linspace(0.4, 4, 10)

for Ψ in range(0, 180, 5):
    
    for r in ri:
        PalaQ_Avanza    = Pala_Avanza(ΩR_Avance, r, V, αR(), βBlc, SBl)
        PalaQ_Retrocede = Pala_Retrocede(ΩR_Avance, r, V, αR(), βBlc, SBl)

        resultados_avanza.append(PalaQ_Avanza)
        resultados_retrocede.append(PalaQ_Retrocede)
        
df_avanza = pd.DataFrame(resultados_avanza, columns=['VBl_A', 'wBl_A', 'ϕBl_A', 'αBl_A', 'CL_A', 'CD_A', 'ABl_A', 'WBl_A',
                                                     'Ms_A', 'FxBl_A', 'MDBl_A', 'Re_A', 'Ma_A', 'FF_A', 'wri_A'])
df_retrocede = pd.DataFrame(resultados_retrocede, columns=['VBl_R', 'wBl_R', 'ϕBl_R', 'αBl_R', 'CL_R', 'CD_R', 'ABl_R', 'WBl_R',
                                                           'Ms_R', 'FxBl_R', 'MDBl_R', 'Re_R', 'Ma_R', 'FF_R', 'wri_A'])
    

print("Resultados para Pala Avanza:")
display(df_avanza.round(3))

print("\nResultados para Pala Retrocede:")
display(df_retrocede.round(3))

# Promedios para la pala que avanza
Re_m_avanza = df_avanza["Re_A"].mean()
CL_m_avanza = df_avanza["CL_A"].mean()
alpha_m_avanza = df_avanza["αBl_A"].mean()

# Promedios para la pala que retrocede
Re_m_retrocede = df_retrocede["Re_R"].mean()
CL_m_retrocede = df_retrocede["CL_R"].mean()
alpha_m_retrocede = df_retrocede["αBl_R"].mean()

# Opcional: Promedio general
Re_medio_total = (Re_m_avanza + Re_m_retrocede) / 2
CL_medio_total = (CL_m_avanza + CL_m_retrocede) / 2
alpha_medio_total = (alpha_m_avanza + alpha_m_retrocede) / 2

print(f"Re medio total: {Re_medio_total:.4f}")
print(f"CL medio total: {CL_medio_total:.4f}")
print(f"Alpha medio total: {alpha_medio_total:.2f}")


Resultados para Pala Avanza:


,VBl_A,wBl_A,ϕBl_A,αBl_A,CL_A,CD_A,ABl_A,WBl_A,Ms_A,FxBl_A,MDBl_A,Re_A,Ma_A,FF_A,wri_A
0,48.6,0.135,0.159,2.659,0.952,0.009,110.225,1.037,44.090,-1.072,58.758,0.666,0.143,295.936,112.771
1,62.2,-0.435,-0.401,2.099,0.955,0.008,181.119,1.547,144.895,-3.530,529.861,0.852,0.183,887.808,11.014
2,75.8,-1.004,-0.759,1.741,0.958,0.008,269.606,2.115,323.527,-6.943,1471.349,1.038,0.223,1479.680,2.678
3,89.4,-1.574,-1.009,1.491,0.957,0.007,374.694,2.928,599.511,-11.281,2882.123,1.225,0.263,2071.552,0.958
4,103.0,-2.144,-1.193,1.307,0.956,0.007,496.921,3.868,993.842,-16.554,4761.054,1.411,0.303,2663.424,0.426
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
355,116.6,-2.714,-1.333,1.167,0.955,0.007,636.239,4.932,1526.974,-22.759,7107.028,1.597,0.343,3255.296,0.218
356,130.2,-3.283,-1.445,1.055,0.954,0.007,792.603,6.120,2219.287,-29.894,9918.933,1.784,0.383,3847.168,0.123
357,143.8,-3.853,-1.535,0.965,0.953,0.007,965.964,7.428,3091.086,-37.956,13195.661,1.970,0.423,4439.040,0.074
358,157.4,-4.423,-1.610,0.890,0.952,0.007,1156.278,8.855,4162.600,-46.942,16936.110,2.156,0.463,5030.912,0.048



Resultados para Pala Retrocede:


,VBl_R,wBl_R,ϕBl_R,αBl_R,CL_R,CD_R,ABl_R,WBl_R,Ms_R,FxBl_R,MDBl_R,Re_R,Ma_R,FF_R,wri_A
0,-21.4,1.274,-3.412,-0.912,0.945,0.013,21.209,0.291,8.484,-1.528,58.576,0.293,-0.063,295.936,-256.107
1,-7.8,1.844,-13.545,-11.045,0.818,0.044,2.439,0.131,1.951,-0.607,532.199,0.107,-0.023,887.808,-87.832
2,5.8,2.414,23.843,26.343,0.584,0.067,0.962,0.111,1.155,0.388,1480.146,0.079,0.017,1479.680,34.998
3,19.4,2.983,8.811,11.311,0.944,0.014,17.410,0.262,27.856,2.460,2904.108,0.266,0.057,2071.552,4.414
4,33.0,3.553,6.169,8.669,0.949,0.010,50.622,0.554,101.244,4.817,4803.798,0.452,0.097,2663.424,1.329
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
355,46.6,4.123,5.069,7.569,0.952,0.009,101.293,0.966,243.102,7.695,7180.119,0.638,0.137,3255.296,0.544
356,60.2,4.692,4.466,6.966,0.955,0.008,169.580,1.470,474.823,11.098,10033.712,0.825,0.177,3847.168,0.265
357,73.8,5.262,4.085,6.585,0.958,0.008,255.600,2.006,817.920,15.029,13365.214,1.011,0.217,4439.040,0.145
358,87.4,5.832,3.823,6.323,0.957,0.007,358.164,2.800,1289.391,19.421,17175.016,1.197,0.257,5030.912,0.086


Re medio total: 1.0647
CL medio total: 0.9281
Alpha medio total: 4.11
